# CrossLead Deeper (3-stage) — Signal Analysis & Threshold Tuning

Loads the recent 3-stage CrossLead model and provides:
1. Train vs test distribution comparison
2. ROC + PR curves (train vs test)
3. **Interactive threshold slider** — drag to find a clinically satisfactory operating point
4. Conv kernels, receptive field, cross-lead attention, saliency

**Model:** `cv_results/repnet_crosslead_deeper_2026-04-26_20-48-58/model_repnet_crosslead_deeper.pt`

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix,
)
from torch.utils.data import DataLoader, TensorDataset

from src.models.repnet_crosslead_deeper import RepNetCrossLeadDeeper
from src.data.dataset import load_seniordesign, split_holdout_grouped
from src.preprocessing.filters import BaselineWanderFilter, NotchFilter
from src.preprocessing.normalization import ZScoreNormalization

C:\Users\email\PycharmProjects\repnet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUN_DIR    = Path('../../cv_results/repnet_crosslead_deeper_2026-04-26_20-48-58')
MODEL_PATH = RUN_DIR / 'model_repnet_crosslead_deeper.pt'
DATA_DIR   = '../../data/seniordesign_upload'
SEED       = 42

NET_PARAMS = dict(
    stage_filters = (32, 64, 128),
    kernels       = (7, 5, 3),
    dropout       = 0.0636,
    n_heads       = 4,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Model:  {MODEL_PATH.resolve()}')

Device: cuda
Model:  C:\Users\email\PycharmProjects\repnet\cv_results\repnet_crosslead_deeper_2026-04-26_20-48-58\model_repnet_crosslead_deeper.pt


In [3]:
net = RepNetCrossLeadDeeper(**NET_PARAMS).to(device)
net.load_state_dict(torch.load(MODEL_PATH, map_location=device))
net.eval()
n_params = sum(p.numel() for p in net.parameters())
print(f'Loaded. Parameters: {n_params:,}')

Loaded. Parameters: 430,466


In [4]:
X, y, patient_ids = load_seniordesign(DATA_DIR, return_patient_ids=True)

flat_mask = (X.std(axis=2) < 1e-4).any(axis=1)
try:
    nan_mask = np.isnan(patient_ids.astype(float))
except (ValueError, TypeError):
    nan_mask = np.array([str(p).strip() in ('', 'nan', 'None') for p in patient_ids])
keep = ~flat_mask & ~nan_mask
X, y, patient_ids = X[keep], y[keep], patient_ids[keep]

X, _ = BaselineWanderFilter(cutoff=0.5, order=4, fs=250.0).transform(X)
X, _ = NotchFilter(freq=60.0, Q=30.0, fs=250.0).transform(X)
X, _ = ZScoreNormalization(per_lead=True).transform(X)

X_dev, X_test, y_dev, y_test, g_dev, g_test = split_holdout_grouped(
    X, y, patient_ids, test_size=0.20, seed=SEED,
)

print(f'Dev (training) : N={len(y_dev)}   PE={int(y_dev.sum())}   Normal={int((y_dev==0).sum())}   ({100*y_dev.mean():.1f}% pos)')
print(f'Test (holdout) : N={len(y_test)}   PE={int(y_test.sum())}   Normal={int((y_test==0).sum())}   ({100*y_test.mean():.1f}% pos)')

Dev (training) : N=1747   PE=277   Normal=1470   (15.9% pos)
Test (holdout) : N=431   PE=58   Normal=373   (13.5% pos)


In [5]:
def infer(net, X, device, batch_size=64):
    Xt = torch.tensor(X, dtype=torch.float32)
    dl = DataLoader(TensorDataset(Xt), batch_size=batch_size, num_workers=0)
    out = []
    with torch.no_grad():
        for (xb,) in dl:
            logits = net(xb.to(device))
            out.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)

probs_dev  = infer(net, X_dev,  device)
probs_test = infer(net, X_test, device)

auroc_dev  = roc_auc_score(y_dev,  probs_dev)
auroc_test = roc_auc_score(y_test, probs_test)
auprc_dev  = average_precision_score(y_dev,  probs_dev)
auprc_test = average_precision_score(y_test, probs_test)

print(f'Train (dev) — AUROC: {auroc_dev:.4f}    AUPRC: {auprc_dev:.4f}')
print(f'Test (held) — AUROC: {auroc_test:.4f}    AUPRC: {auprc_test:.4f}')
print(f'Generalization gap: {auroc_dev - auroc_test:+.4f}')

Train (dev) — AUROC: 0.8571    AUPRC: 0.5412
Test (held) — AUROC: 0.6842    AUPRC: 0.2117
Generalization gap: +0.1728


## Train vs test distribution

In [6]:
# Mirror histogram: PE up, Normal down. Easier to read separation at a glance.
def mirror_hist_traces(probs_, y_, nbins=40, show_legend=False):
    bins = np.linspace(0, 1, nbins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    width = bins[1] - bins[0]
    h_norm, _ = np.histogram(probs_[y_ == 0], bins=bins, density=True)
    h_pe,   _ = np.histogram(probs_[y_ == 1], bins=bins, density=True)
    return [
        go.Bar(x=centers, y=h_pe, name='PE',
               marker_color='tomato', width=width,
               showlegend=show_legend, legendgroup='PE'),
        go.Bar(x=centers, y=-h_norm, name='Normal',
               marker_color='steelblue', width=width,
               showlegend=show_legend, legendgroup='Normal'),
    ], h_norm.max(), h_pe.max()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=(f'Train  AUROC={auroc_dev:.3f}  N={len(y_dev)}',
                    f'Test   AUROC={auroc_test:.3f}  N={len(y_test)}'))

ymax = 0.0
for col, (probs_, y_) in enumerate([(probs_dev, y_dev), (probs_test, y_test)], start=1):
    traces, hn, hp = mirror_hist_traces(probs_, y_, show_legend=(col == 1))
    ymax = max(ymax, hn, hp)
    for tr in traces:
        fig.add_trace(tr, row=1, col=col)
    fig.add_vline(x=0.5, line=dict(dash='dash', color='black'), row=1, col=col)

# Symmetric y-axis with absolute-value tick labels
ymax = ymax * 1.1
tickvals = np.linspace(-ymax, ymax, 7)
ticktext = [f'{abs(v):.1f}' for v in tickvals]
fig.update_yaxes(tickvals=tickvals, ticktext=ticktext,
                 zeroline=True, zerolinecolor='black', zerolinewidth=1)

fig.update_layout(template='plotly_white', barmode='overlay', bargap=0,
                  title='P(PE) — train vs test (PE ↑, Normal ↓)',
                  width=1100, height=460)
fig.update_xaxes(title_text='P(PE)')
fig.update_yaxes(title_text='Density (PE up / Normal down)', col=1)
fig.show()

## ROC + Precision-Recall (train vs test)

In [7]:
fpr_dv, tpr_dv, _ = roc_curve(y_dev,  probs_dev)
fpr_te, tpr_te, _ = roc_curve(y_test, probs_test)
prec_dv, rec_dv, _ = precision_recall_curve(y_dev,  probs_dev)
prec_te, rec_te, _ = precision_recall_curve(y_test, probs_test)

fig2 = make_subplots(rows=1, cols=2, subplot_titles=('ROC', 'Precision-Recall'))
fig2.add_trace(go.Scatter(x=fpr_dv, y=tpr_dv, name=f'Train (AUC={auroc_dev:.3f})',
                          line=dict(color='steelblue')), row=1, col=1)
fig2.add_trace(go.Scatter(x=fpr_te, y=tpr_te, name=f'Test (AUC={auroc_test:.3f})',
                          line=dict(color='tomato')), row=1, col=1)
fig2.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='Random',
                          line=dict(dash='dash', color='gray'), showlegend=False), row=1, col=1)
fig2.add_trace(go.Scatter(x=rec_dv, y=prec_dv, name=f'Train (AP={auprc_dev:.3f})',
                          line=dict(color='steelblue', dash='dot')), row=1, col=2)
fig2.add_trace(go.Scatter(x=rec_te, y=prec_te, name=f'Test (AP={auprc_test:.3f})',
                          line=dict(color='tomato', dash='dot')), row=1, col=2)

fig2.update_xaxes(title_text='FPR', row=1, col=1)
fig2.update_yaxes(title_text='TPR', row=1, col=1)
fig2.update_xaxes(title_text='Recall', row=1, col=2)
fig2.update_yaxes(title_text='Precision', row=1, col=2)
fig2.update_layout(template='plotly_white', width=1100, height=480,
                   title='ROC & PR — train vs test')
fig2.show()

## Interactive threshold slider (test set)

Drag τ to update the confusion matrix and metric panel.  
For preeclampsia screening, prioritize sensitivity (≥0.85) since false negatives are clinically costly.  
PPV will be naturally low because of the ~13% positive rate.

In [8]:
def metrics_at(probs_, y_, tau):
    pred = (probs_ >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_, pred, labels=[0, 1]).ravel()
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    ppv  = tp / max(tp + fp, 1)
    npv  = tn / max(tn + fn, 1)
    f1   = 2 * ppv * sens / max(ppv + sens, 1e-9)
    acc  = (tp + tn) / (tp + tn + fp + fn)
    return dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                sens=sens, spec=spec, ppv=ppv, npv=npv, f1=f1, acc=acc)

In [9]:
thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
metric_table = [metrics_at(probs_test, y_test, t) for t in thresholds]

p_norm = probs_test[y_test == 0]
p_pe   = probs_test[y_test == 1]

# Mirror histogram (PE up, Normal down)
nbins = 40
bins = np.linspace(0, 1, nbins + 1)
centers = 0.5 * (bins[:-1] + bins[1:])
bar_w   = bins[1] - bins[0]
h_norm, _ = np.histogram(p_norm, bins=bins, density=True)
h_pe,   _ = np.histogram(p_pe,   bins=bins, density=True)

fig3 = make_subplots(
    rows=1, cols=2,
    column_widths=[0.62, 0.38],
    specs=[[{'type': 'xy'}, {'type': 'heatmap'}]],
    subplot_titles=('Test P(PE) distribution  (PE ↑, Normal ↓)', 'Confusion matrix'),
)

fig3.add_trace(go.Bar(x=centers, y=h_pe, name='PE',
                      marker_color='tomato', width=bar_w), row=1, col=1)
fig3.add_trace(go.Bar(x=centers, y=-h_norm, name='Normal',
                      marker_color='steelblue', width=bar_w), row=1, col=1)

init_tau = 0.50
init_idx = int(np.argmin(np.abs(thresholds - init_tau)))
init_m   = metric_table[init_idx]

cm_z = [[init_m['tn'], init_m['fp']], [init_m['fn'], init_m['tp']]]
fig3.add_trace(go.Heatmap(
    z=cm_z, x=['Pred Normal', 'Pred PE'], y=['True Normal', 'True PE'],
    colorscale='Blues', showscale=False,
    text=cm_z, texttemplate='%{text}', textfont={'size': 18},
), row=1, col=2)

fig3.add_shape(type='line', x0=init_tau, x1=init_tau, y0=0, y1=1,
               yref='paper', xref='x', line=dict(color='black', dash='dash', width=2))

def metric_text(t, m):
    return (f'<b>τ = {t:.2f}</b><br>'
            f'Sensitivity : {m["sens"]:.3f}<br>'
            f'Specificity : {m["spec"]:.3f}<br>'
            f'PPV         : {m["ppv"]:.3f}<br>'
            f'NPV         : {m["npv"]:.3f}<br>'
            f'F1          : {m["f1"]:.3f}<br>'
            f'Accuracy    : {m["acc"]:.3f}')

fig3.add_annotation(
    text=metric_text(init_tau, init_m),
    xref='paper', yref='paper', x=0.40, y=0.98,
    align='left', showarrow=False,
    bgcolor='rgba(255,255,255,0.85)',
    bordercolor='gray', borderwidth=1, borderpad=8,
    font=dict(family='monospace', size=12),
)

steps = []
for i, t in enumerate(thresholds):
    m = metric_table[i]
    cm_new = [[m['tn'], m['fp']], [m['fn'], m['tp']]]
    steps.append(dict(
        method='update',
        label=f'{t:.2f}',
        args=[
            {'z': [None, None, [cm_new]],
             'text': [None, None, [cm_new]]},
            {'shapes': [dict(type='line', x0=float(t), x1=float(t), y0=0, y1=1,
                             yref='paper', xref='x',
                             line=dict(color='black', dash='dash', width=2))],
             'annotations[2].text': metric_text(t, m)},
        ],
    ))

ymax = max(h_pe.max(), h_norm.max()) * 1.1
tickvals = np.linspace(-ymax, ymax, 7)
ticktext = [f'{abs(v):.1f}' for v in tickvals]

fig3.update_layout(
    sliders=[dict(active=init_idx, currentvalue={'prefix': 'τ = '},
                  pad={'t': 50}, steps=steps)],
    barmode='overlay', bargap=0, template='plotly_white',
    width=1200, height=560,
    title=f'Threshold tuner — CrossLead Deeper (test, AUROC={auroc_test:.3f})',
)
fig3.update_xaxes(title_text='P(PE)', row=1, col=1)
fig3.update_yaxes(title_text='Density (PE up / Normal down)',
                  tickvals=tickvals, ticktext=ticktext,
                  zeroline=True, zerolinecolor='black', zerolinewidth=1,
                  row=1, col=1)
fig3.show()

## Threshold sweep table

In [10]:
key_taus = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
rows = []
for t in key_taus:
    m = metrics_at(probs_test, y_test, t)
    rows.append({'τ': t, 'TP': m['tp'], 'FP': m['fp'], 'FN': m['fn'], 'TN': m['tn'],
                 'Sens': round(m['sens'], 3), 'Spec': round(m['spec'], 3),
                 'PPV': round(m['ppv'], 3), 'NPV': round(m['npv'], 3),
                 'F1': round(m['f1'], 3), 'Acc': round(m['acc'], 3)})
pd.DataFrame(rows).set_index('τ')

,TP,FP,FN,TN,Sens,Spec,PPV,NPV,F1,Acc
τ,,,,,,,,,,
0.1,58,349,0,24,1.000,0.064,0.143,1.000,0.249,0.190
0.2,57,330,1,43,0.983,0.115,0.147,0.977,0.256,0.232
0.3,57,311,1,62,0.983,0.166,0.155,0.984,0.268,0.276
0.4,57,293,1,80,0.983,0.214,0.163,0.988,0.279,0.318
0.5,55,259,3,114,0.948,0.306,0.175,0.974,0.296,0.392
0.6,54,241,4,132,0.931,0.354,0.183,0.971,0.306,0.432
0.7,52,213,6,160,0.897,0.429,0.196,0.964,0.322,0.492
0.8,43,177,15,196,0.741,0.525,0.195,0.929,0.309,0.555


## Find threshold meeting clinical targets

In [11]:
def threshold_for_target(probs_, y_, target_sens=0.85):
    fpr, tpr, thr = roc_curve(y_, probs_)
    valid = tpr >= target_sens
    if not valid.any():
        return None
    i = int(np.argmax(thr[valid]))
    return float(thr[valid][i])

for target in [0.95, 0.90, 0.85, 0.80]:
    t = threshold_for_target(probs_test, y_test, target)
    if t is None:
        print(f'target sens {target}: not achievable')
        continue
    m = metrics_at(probs_test, y_test, t)
    print(f'sens >= {target} → τ={t:.3f}  sens={m["sens"]:.3f}  spec={m["spec"]:.3f}  '
          f'PPV={m["ppv"]:.3f}  F1={m["f1"]:.3f}  (TP={m["tp"]}, FP={m["fp"]}, FN={m["fn"]}, TN={m["tn"]})')

sens >= 0.95 → τ=0.480  sens=0.966  spec=0.279  PPV=0.172  F1=0.292  (TP=56, FP=269, FN=2, TN=104)
sens >= 0.9 → τ=0.684  sens=0.914  spec=0.421  PPV=0.197  F1=0.324  (TP=53, FP=216, FN=5, TN=157)
sens >= 0.85 → τ=0.725  sens=0.862  spec=0.448  PPV=0.195  F1=0.318  (TP=50, FP=206, FN=8, TN=167)
sens >= 0.8 → τ=0.766  sens=0.810  spec=0.485  PPV=0.197  F1=0.316  (TP=47, FP=192, FN=11, TN=181)


---

# Feature-detection diagnostics

1. **Conv kernels** — what features the first stage learned to detect
2. **Receptive field** — how much temporal context each output unit sees
3. **Cross-lead attention** — which leads the model focuses on, per class
4. **Saliency maps** — gradient of P(PE) w.r.t. each input sample, overlaid on the ECG

## 1. First-stage conv kernels

32 stage-1 kernels (k=7 = 28 ms @ 250 Hz). Look for derivative/peak/oscillatory shapes —  
noise-like shapes mean the model failed to learn meaningful low-level features.

In [12]:
k1 = net.stages[0]['conv'].conv1.weight.detach().cpu().squeeze(1).numpy()  # (32, 7)
order = np.argsort(-np.linalg.norm(k1, axis=1))
k1_sorted = k1[order]

fig_k = go.Figure(data=go.Heatmap(
    z=k1_sorted, colorscale='RdBu_r', zmid=0,
    colorbar=dict(title='weight'),
))
fig_k.update_layout(
    title='Stage 1 conv1 kernels (32 filters × 7 taps), sorted by L2 norm',
    xaxis_title='kernel tap (4 ms each)',
    yaxis_title='filter idx (sorted)',
    template='plotly_white', width=620, height=560,
)
fig_k.show()

fig_kl = go.Figure()
for i in range(12):
    fig_kl.add_trace(go.Scatter(
        y=k1_sorted[i], mode='lines+markers',
        name=f'filter {order[i]} (||·||={np.linalg.norm(k1_sorted[i]):.2f})',
    ))
fig_kl.update_layout(title='Top-12 stage-1 kernels by L2 norm',
                     xaxis_title='tap', yaxis_title='weight',
                     template='plotly_white', width=900, height=420)
fig_kl.show()

## 2. Receptive field analysis

In [13]:
def rf_table(kernels, fs=250.0):
    rf, eff_stride = 1, 1
    rows = []
    for stage_idx, k in enumerate(kernels, start=1):
        rf += (k - 1) * eff_stride
        rows.append({'stage': stage_idx, 'layer': 'conv1', 'k': k, 'RF (samples)': rf,
                     'RF (ms)': round(1000*rf/fs, 1)})
        rf += (k - 1) * eff_stride
        rows.append({'stage': stage_idx, 'layer': 'conv2', 'k': k, 'RF (samples)': rf,
                     'RF (ms)': round(1000*rf/fs, 1)})
        eff_stride *= 2
        rows.append({'stage': stage_idx, 'layer': 'pool', 'k': '-', 'RF (samples)': rf,
                     'RF (ms)': round(1000*rf/fs, 1)})
    return pd.DataFrame(rows)

rf_df = rf_table(NET_PARAMS['kernels'], fs=250.0)
print(f"Total RF: {rf_df['RF (samples)'].iloc[-1]} samples = "
      f"{rf_df['RF (ms)'].iloc[-1]} ms @ 250 Hz")
rf_df

Total RF: 45 samples = 180.0 ms @ 250 Hz


,stage,layer,k,RF (samples),RF (ms)
0,1,conv1,7,7,28.0
1,1,conv2,7,13,52.0
2,1,pool,-,13,52.0
3,2,conv1,5,21,84.0
4,2,conv2,5,29,116.0
5,2,pool,-,29,116.0
6,3,conv1,3,37,148.0
7,3,conv2,3,45,180.0
8,3,pool,-,45,180.0


In [14]:
# Empirical RF: gradient of a center output unit w.r.t. input.
net.zero_grad()
sample = torch.tensor(X_test[0:1], dtype=torch.float32, device=device, requires_grad=True)

captured = {}
def cap_hook(name):
    def h(_m, _i, o): captured[name] = o
    return h
h = net.stages[-1]['conv'].register_forward_hook(cap_hook('post_conv'))
_ = net(sample)
h.remove()

feat = captured['post_conv']
B, L, C, T_out = feat.shape
center_t = T_out // 2

target = feat[0, :, :, center_t].abs().sum()
grad = torch.autograd.grad(target, sample)[0][0].abs().cpu().numpy()  # (12, 2500)

erf_per_t = grad.mean(axis=0)
nonzero = erf_per_t > erf_per_t.max() * 0.01
erf_extent = int(np.ptp(np.where(nonzero)[0]) if nonzero.any() else 0)
print(f'Empirical RF (>1% of peak): {erf_extent} samples = {1000*erf_extent/250:.1f} ms')
print(f'Analytical RF: {rf_df["RF (samples)"].iloc[-1]} samples = {rf_df["RF (ms)"].iloc[-1]} ms')

half = 100
center_input = (center_t * 8) + 4
center_input = min(max(center_input, half), 2500 - half)
window = slice(center_input - half, center_input + half)
fig_erf = go.Figure()
fig_erf.add_trace(go.Scatter(y=erf_per_t[window], mode='lines',
                              line=dict(color='steelblue'), name='|grad|'))
fig_erf.update_layout(title='Effective receptive field (centered) — sample 0',
                      xaxis_title='input sample (relative to center)',
                      yaxis_title='|∂feat/∂input| (mean over leads)',
                      template='plotly_white', width=820, height=380)
fig_erf.show()

Empirical RF (>1% of peak): 38 samples = 152.0 ms
Analytical RF: 45 samples = 180.0 ms


## 3. Cross-lead attention

Captures (12 × 12) attention matrices from each `MultiheadAttention` via forward hooks.

In [15]:
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
attn_storage = {f'stage{i+1}': [] for i in range(len(net.stages))}

def make_attn_hook(name):
    def hook(module, inputs, output):
        if isinstance(output, tuple) and len(output) >= 2 and output[1] is not None:
            attn_storage[name].append(output[1].detach().cpu().numpy())
    return hook

handles = []
for i, stage in enumerate(net.stages):
    handles.append(stage['attn'].attn.register_forward_hook(make_attn_hook(f'stage{i+1}')))

with torch.no_grad():
    Xt = torch.tensor(X_test, dtype=torch.float32)
    dl = DataLoader(TensorDataset(Xt), batch_size=64)
    for (xb,) in dl:
        net(xb.to(device))

for h in handles: h.remove()

attn_by_stage = {k: np.concatenate(v, axis=0) for k, v in attn_storage.items()}
for k, v in attn_by_stage.items():
    print(f'{k}: shape={v.shape}')

stage1: shape=(431, 12, 12)
stage2: shape=(431, 12, 12)
stage3: shape=(431, 12, 12)


In [16]:
n_stages = len(attn_by_stage)
fig_a = make_subplots(rows=2, cols=n_stages,
    subplot_titles=[f'Stage {i+1} — Normal' for i in range(n_stages)] +
                   [f'Stage {i+1} — PE' for i in range(n_stages)],
    horizontal_spacing=0.08, vertical_spacing=0.12)

for col, k in enumerate([f'stage{i+1}' for i in range(n_stages)], start=1):
    A = attn_by_stage[k]
    A_norm = A[y_test == 0].mean(axis=0)
    A_pe   = A[y_test == 1].mean(axis=0)
    fig_a.add_trace(go.Heatmap(z=A_norm, x=LEADS, y=LEADS,
                               colorscale='Blues', showscale=(col == n_stages),
                               colorbar=dict(x=1.02, len=0.4, y=0.78)),
                    row=1, col=col)
    fig_a.add_trace(go.Heatmap(z=A_pe, x=LEADS, y=LEADS,
                               colorscale='Reds', showscale=(col == n_stages),
                               colorbar=dict(x=1.02, len=0.4, y=0.22)),
                    row=2, col=col)

fig_a.update_layout(template='plotly_white', width=1200, height=700,
                    title='Mean cross-lead attention per stage (rows=query, cols=key)')
fig_a.show()

In [17]:
fig_li = make_subplots(rows=1, cols=n_stages,
                       subplot_titles=[f'Stage {i+1}' for i in range(n_stages)])
for col, k in enumerate([f'stage{i+1}' for i in range(n_stages)], start=1):
    A = attn_by_stage[k]
    incoming_norm = A[y_test == 0].mean(axis=0).sum(axis=1)
    incoming_pe   = A[y_test == 1].mean(axis=0).sum(axis=1)
    fig_li.add_trace(go.Bar(x=LEADS, y=incoming_norm, name='Normal',
                            marker_color='steelblue', showlegend=(col == 1)),
                     row=1, col=col)
    fig_li.add_trace(go.Bar(x=LEADS, y=incoming_pe, name='PE',
                            marker_color='tomato', showlegend=(col == 1)),
                     row=1, col=col)
fig_li.update_layout(barmode='group', template='plotly_white',
                     title='Lead-level attention (row sum) — Normal vs PE',
                     width=1200, height=380)
fig_li.show()

## 4. Saliency intervals (Integrated Gradients)

**Method:** Integrated Gradients with thresholded interval extraction.
1. Compute IG attribution: `(x − 0) · ∫ ∂y/∂x dα` along path from zero baseline to input
2. Smooth with Gaussian σ=8 samples (≈32 ms)
3. Find contiguous time intervals where `|attribution|` exceeds 40% of the global max
4. Render those intervals as colored bands on the ECG: **red = pushes toward PE, blue = pushes toward Normal**

**Four cases** to compare what the model uses across confidence levels:
- **Confident TRUE** — most certain *correct* prediction (highest `|P − 0.5|` among correct)
- **Confident FALSE** — most certain *wrong* prediction (highest `|P − 0.5|` among wrong) — *the failure mode the model commits to*
- **Borderline TRUE** — correct, just barely (lowest `|P − 0.5|` among correct) — *what tipped it the right way*
- **Borderline FALSE** — wrong, just barely (lowest `|P − 0.5|` among wrong) — *what misled it*

In [18]:
from scipy.ndimage import gaussian_filter1d


def integrated_gradients(net, x_np, target_class=1, n_steps=32, baseline=None,
                         smooth_sigma=5.0):
    """Signed (12, T) IG attribution. Positive = pushes toward target_class."""
    x = torch.tensor(x_np, dtype=torch.float32, device=device)
    base = (torch.zeros_like(x) if baseline is None
            else torch.tensor(baseline, dtype=torch.float32, device=device))

    alphas = torch.linspace(0.5 / n_steps, 1.0 - 0.5 / n_steps, n_steps,
                            device=device).view(-1, 1, 1)
    interp = base.unsqueeze(0) + alphas * (x - base).unsqueeze(0)
    interp.requires_grad_(True)

    net.zero_grad()
    logits = net(interp)
    grads = torch.autograd.grad(logits[:, target_class].sum(), interp)[0]

    avg_grad   = grads.mean(dim=0)
    attribution = ((x - base) * avg_grad).cpu().numpy()
    if smooth_sigma > 0:
        attribution = gaussian_filter1d(attribution, sigma=smooth_sigma, axis=1)
    return attribution


def extract_intervals(attr, threshold_frac=0.20, min_samples=5, fs=250.0,
                      per_lead_threshold=True):
    abs_attr = np.abs(attr)
    intervals = []
    for L in range(attr.shape[0]):
        lead_max = float(abs_attr[L].max())
        if lead_max == 0:
            continue
        ref = lead_max if per_lead_threshold else float(abs_attr.max())
        thresh = ref * threshold_frac
        above = abs_attr[L] > thresh
        if not above.any():
            continue
        diff = np.diff(above.astype(int), prepend=0, append=0)
        starts = np.where(diff == 1)[0]
        ends   = np.where(diff == -1)[0]
        for s, e in zip(starts, ends):
            if (e - s) < min_samples:
                continue
            intervals.append({
                'lead': L,
                't_start': s / fs, 't_end': e / fs,
                's_idx': int(s), 'e_idx': int(e),
                'sign':      float(attr[L, s:e].mean()),
                'magnitude': float(abs_attr[L, s:e].mean()),
                'duration_ms': (e - s) * 1000.0 / fs,
            })
    return intervals


def pick_8_cases(probs, y):
    """Pick 8 examples covering {confident, borderline} × {TP, TN, FP, FN}.

    Always splits by class so a confident PE (TP) is guaranteed to appear,
    independent of class balance.
    """
    pred = (probs >= 0.5).astype(int)

    # Per-class, per-correctness index pools
    tp_idx = np.where((y == 1) & (pred == 1))[0]   # PE called PE
    tn_idx = np.where((y == 0) & (pred == 0))[0]   # Normal called Normal
    fp_idx = np.where((y == 0) & (pred == 1))[0]   # Normal called PE
    fn_idx = np.where((y == 1) & (pred == 0))[0]   # PE called Normal

    out = {}

    # CONFIDENT (extreme P) — TP highest, TN lowest, FP highest, FN lowest
    if len(tp_idx): out['confident_TP'] = tp_idx[np.argmax(probs[tp_idx])]
    if len(tn_idx): out['confident_TN'] = tn_idx[np.argmin(probs[tn_idx])]
    if len(fp_idx): out['confident_FP'] = fp_idx[np.argmax(probs[fp_idx])]
    if len(fn_idx): out['confident_FN'] = fn_idx[np.argmin(probs[fn_idx])]

    # BORDERLINE — closest to 0.5 within each pool
    if len(tp_idx): out['borderline_TP'] = tp_idx[np.argmin(np.abs(probs[tp_idx] - 0.5))]
    if len(tn_idx): out['borderline_TN'] = tn_idx[np.argmin(np.abs(probs[tn_idx] - 0.5))]
    if len(fp_idx): out['borderline_FP'] = fp_idx[np.argmin(np.abs(probs[fp_idx] - 0.5))]
    if len(fn_idx): out['borderline_FN'] = fn_idx[np.argmin(np.abs(probs[fn_idx] - 0.5))]

    return out


cases = pick_8_cases(probs_test, y_test)
print('Selected examples:')
print(f'  {"category":<18}  {"idx":>4}  {"true":>4}  {"P(PE)":>6}  {"meaning":<30}')
meanings = {
    'confident_TP':  'PE correctly called (high P)',
    'confident_TN':  'Normal correctly called (low P)',
    'confident_FP':  'Normal called PE (high P, WRONG)',
    'confident_FN':  'PE called Normal (low P, WRONG)',
    'borderline_TP': 'PE correctly called, near 0.5',
    'borderline_TN': 'Normal correctly called, near 0.5',
    'borderline_FP': 'Normal called PE, near 0.5',
    'borderline_FN': 'PE called Normal, near 0.5',
}
for label, idx in cases.items():
    print(f'  {label:<18}  {idx:>4}  {int(y_test[idx]):>4}  {probs_test[idx]:>6.3f}  {meanings[label]:<30}')

# Compute attributions and intervals for all selected cases
attributions = {idx: integrated_gradients(net, X_test[idx], target_class=1)
                for idx in cases.values()}
intervals_by_idx = {idx: extract_intervals(attributions[idx])
                    for idx in cases.values()}

print('\nInterval summary:')
for label, idx in cases.items():
    ivs = intervals_by_idx[idx]
    leads_with = sorted(set(iv['lead'] for iv in ivs))
    leads_str = ', '.join(LEADS[L] for L in leads_with) if leads_with else '(none)'
    print(f'  {label:<18}  {len(ivs):3d} intervals across {len(leads_with):2d} leads  ({leads_str})')

Selected examples:
  category             idx  true   P(PE)  meaning                       
  confident_TP         385     1   0.994  PE correctly called (high P)  
  confident_TN         221     0   0.002  Normal correctly called (low P)
  confident_FP          96     0   0.999  Normal called PE (high P, WRONG)
  confident_FN         406     1   0.177  PE called Normal (low P, WRONG)
  borderline_TP        404     1   0.562  PE correctly called, near 0.5 
  borderline_TN        182     0   0.496  Normal correctly called, near 0.5
  borderline_FP        304     0   0.502  Normal called PE, near 0.5    
  borderline_FN        409     1   0.480  PE called Normal, near 0.5    

Interval summary:
  confident_TP        458 intervals across 12 leads  (I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6)
  confident_TN        445 intervals across 12 leads  (I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6)
  confident_FP        451 intervals across 12 leads  (I, II, III, aVR, aVL, aVF, V1, V2,

In [19]:
# Per-lead net signed attribution per case
ordered = list(cases.items())
fig_lead = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'{label} | idx={idx} | true={int(y_test[idx])} | P(PE)={probs_test[idx]:.3f}'
        for label, idx in ordered
    ],
    vertical_spacing=0.18,
)
for n, (label, idx) in enumerate(ordered):
    a = attributions[idx]
    net_sign = a.sum(axis=1)
    colors = ['tomato' if v > 0 else 'steelblue' for v in net_sign]
    r, c = n // 2 + 1, n % 2 + 1
    fig_lead.add_trace(go.Bar(x=LEADS, y=net_sign, marker_color=colors,
                              showlegend=False), row=r, col=c)
    fig_lead.add_hline(y=0, line=dict(color='black', width=1), row=r, col=c)

fig_lead.update_layout(
    template='plotly_white', width=1200, height=620,
    title='Net signed attribution per lead — 4 cases  '
          '(red = pushes toward PE, blue = toward Normal)',
)
fig_lead.show()

Exception: The (row, col) pair sent is out of range. Use Figure.print_grid to view the subplot grid. 

In [ ]:
# ECG with discrete attribution intervals as colored bands behind the trace.
# Uses add_shape with explicit xref/yref instead of add_vrect — more reliable in subplots.

def plot_intervals_for_case(label, idx, top_k=4, fs=250.0):
    attr      = attributions[idx]
    intervals = intervals_by_idx[idx]
    ecg       = X_test[idx]
    t         = np.arange(ecg.shape[1]) / fs

    # Rank leads by total interval "energy" (magnitude × duration). Fallback to total |attr|.
    by_lead = {}
    for iv in intervals:
        by_lead[iv['lead']] = by_lead.get(iv['lead'], 0.0) + iv['magnitude'] * (iv['e_idx'] - iv['s_idx'])
    if by_lead:
        top_leads = sorted(by_lead, key=lambda L: -by_lead[L])[:top_k]
    else:
        top_leads = list(np.argsort(-np.abs(attr).sum(axis=1))[:top_k])

    fig = make_subplots(
        rows=top_k, cols=1, shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=[f'Lead {LEADS[L]}' for L in top_leads],
    )

    # First add ECG traces (so we know the y-range per subplot).
    for r, L in enumerate(top_leads, start=1):
        fig.add_trace(go.Scatter(
            x=t, y=ecg[L], mode='lines',
            line=dict(color='black', width=1.2), showlegend=False,
        ), row=r, col=1)

    # Now add interval rectangles. Use explicit xref/yref keyed to each subplot.
    # In a single-column subplot grid, axes are x, x2, x3, ... and y, y2, y3, ...
    # plotly indexes from 1 with no suffix on the first axis.
    shapes = []
    for r, L in enumerate(top_leads, start=1):
        xref = 'x'  if r == 1 else f'x{r}'
        yref = 'y'  if r == 1 else f'y{r}'
        ymin, ymax = float(ecg[L].min()), float(ecg[L].max())
        # Pad y a bit so bands extend just beyond the trace
        ypad = (ymax - ymin) * 0.1
        for iv in intervals:
            if iv['lead'] != L:
                continue
            color = ('rgba(220,70,70,0.40)' if iv['sign'] > 0
                     else 'rgba(70,130,180,0.40)')
            shapes.append(dict(
                type='rect',
                x0=iv['t_start'], x1=iv['t_end'],
                y0=ymin - ypad,  y1=ymax + ypad,
                xref=xref, yref=yref,
                fillcolor=color, layer='below', line=dict(width=0),
            ))

    pred       = int(probs_test[idx] >= 0.5)
    true_label = 'PE' if y_test[idx] == 1 else 'Normal'
    pred_label = 'PE' if pred == 1 else 'Normal'
    correct    = pred == int(y_test[idx])

    fig.update_layout(
        template='plotly_white', width=1200, height=180 * top_k + 100,
        shapes=shapes,
        title=(f'<b>{label.replace("_", " ")}</b>  |  idx {idx}  |  '
               f'true={true_label}  pred={pred_label}  P(PE)={probs_test[idx]:.3f}  |  '
               f'{"correct" if correct else "WRONG"}<br>'
               f'<sub>red bands = pushes toward PE, blue = toward Normal  |  '
               f'{len(intervals)} total intervals on {len(by_lead) if by_lead else 0} leads</sub>'),
    )
    fig.update_xaxes(title_text='time (s)', row=top_k, col=1)
    return fig


# Sanity check: print intervals on the top lead of each case so we can confirm extraction works
print('Intervals on top-attribution lead per case:')
for label, idx in ordered:
    ivs = intervals_by_idx[idx]
    if not ivs:
        print(f'  {label:<18}  (no intervals — bands will not appear)')
        continue
    by_lead = {}
    for iv in ivs:
        by_lead[iv['lead']] = by_lead.get(iv['lead'], 0.0) + iv['magnitude']
    top = max(by_lead, key=by_lead.get)
    top_ivs = [iv for iv in ivs if iv['lead'] == top]
    bands = ', '.join(f'{iv["t_start"]:.2f}-{iv["t_end"]:.2f}s({"+%.0f" if iv["sign"]>0 else "-%.0f"})'
                      .format(abs(iv["sign"]) * 1000)
                      for iv in top_ivs[:5])
    print(f'  {label:<18}  top lead {LEADS[top]}: {bands}')

print()
for label, idx in ordered:
    plot_intervals_for_case(label, idx, top_k=4).show()

## Reading the diagnostics

- **Kernels**: structured shapes (peaks, derivatives, oscillations) → real low-level features.  
  Pure noise → kernels weren't usefully trained.  
- **Receptive field**: empirical RF should match the analytical bound, or be smaller.  
- **Cross-lead attention**: if Normal and PE matrices look identical, attention isn't class-discriminative.  
  If a few leads dominate (e.g. V2–V4), the model has learned a useful prior.  
- **Saliency**: should concentrate on QRS / T-wave regions. If it's uniform or focused on  
  signal edges, the model is using artifacts, not physiology.

A model with `train_AUROC=0.857, test_AUROC=0.684` (gap +0.17) is overfitting — these diagnostics  
should help distinguish whether it's overfitting to *meaningful* features (need more data) or to  
*noise* (need different architecture or augmentation).